In [128]:
from nse_platform.context.execution_context import ExecutionContext
from pyspark.sql import functions as F

ec = ExecutionContext.for_local_spark("explore", "2026-01-01")
spark = ec.spark

In [129]:
df = spark.read.parquet("../data/bronze/index/nse/year=2026/month=01/day=01")
df.printSchema()
df.show(20)
print(df.columns)

root
 |-- Index Name: string (nullable = true)
 |-- Index Date: string (nullable = true)
 |-- Open Index Value: string (nullable = true)
 |-- High Index Value: string (nullable = true)
 |-- Low Index Value: string (nullable = true)
 |-- Closing Index Value: string (nullable = true)
 |-- Points Change: string (nullable = true)
 |-- Change(%): string (nullable = true)
 |-- Volume: string (nullable = true)
 |-- Turnover (Rs. Cr.): string (nullable = true)
 |-- P/E: string (nullable = true)
 |-- P/B: string (nullable = true)
 |-- Div Yield: string (nullable = true)

+--------------------+----------+----------------+----------------+---------------+-------------------+-------------+---------+----------+------------------+-----+----+---------+
|          Index Name|Index Date|Open Index Value|High Index Value|Low Index Value|Closing Index Value|Points Change|Change(%)|    Volume|Turnover (Rs. Cr.)|  P/E| P/B|Div Yield|
+--------------------+----------+----------------+----------------+------

In [130]:
map_columns = {"Index Name": "index_name",
               "Index Date": "trade_date",
               "Open Index Value": "open",
               "High Index Value": "high",
               "Low Index Value": "low",
               "Closing Index Value": "close",
               "Volume":"volume",
               "Turnover (Rs. Cr.)":"turnover_in_rs_cr",
               "P/E":"pe",
               "P/B":"pb",
               "Div Yield": "div_yield"
               }

In [131]:
df = df.withColumnsRenamed(map_columns).select([c for c in map_columns.values()])
df.show(10, truncate=False)

+-----------------------+----------+--------+--------+--------+--------+----------+-----------------+-----+----+---------+
|index_name             |trade_date|open    |high    |low     |close   |volume    |turnover_in_rs_cr|pe   |pb  |div_yield|
+-----------------------+----------+--------+--------+--------+--------+----------+-----------------+-----+----+---------+
|Nifty 50               |01-01-2026|26173.3 |26197.55|26113.4 |26146.55|425631910 |23454.66         |22.76|3.56|1.28     |
|Nifty Next 50          |01-01-2026|69523.15|69722.15|69276.8 |69675.4 |216114798 |7311.28          |20.27|3.62|1.41     |
|Nifty 100              |01-01-2026|26736.55|26764.25|26674.1 |26723.25|641746708 |30765.95         |22.32|3.57|1.3      |
|Nifty 200              |01-01-2026|14560.7 |14571.25|14518.4 |14560.25|3987031128|47016.89         |23.81|3.7 |1.21     |
|Nifty 500              |01-01-2026|23914.65|23925.55|23835.55|23909.55|4549066214|61090.24         |24.49|3.72|1.15     |
|Nifty Midcap 50

In [132]:
df.filter(F.col("open") == "-").count()

12

In [133]:
df = df.withColumn("index_date", F.to_date("trade_date","dd-MM-yyyy")) \
        .withColumn("volume", F.col("volume").try_cast("long"))

In [134]:
df.show(10)

+--------------------+----------+--------+--------+--------+--------+----------+-----------------+-----+----+---------+----------+
|          index_name|trade_date|    open|    high|     low|   close|    volume|turnover_in_rs_cr|   pe|  pb|div_yield|index_date|
+--------------------+----------+--------+--------+--------+--------+----------+-----------------+-----+----+---------+----------+
|            Nifty 50|01-01-2026| 26173.3|26197.55| 26113.4|26146.55| 425631910|         23454.66|22.76|3.56|     1.28|2026-01-01|
|       Nifty Next 50|01-01-2026|69523.15|69722.15| 69276.8| 69675.4| 216114798|          7311.28|20.27|3.62|     1.41|2026-01-01|
|           Nifty 100|01-01-2026|26736.55|26764.25| 26674.1|26723.25| 641746708|         30765.95|22.32|3.57|      1.3|2026-01-01|
|           Nifty 200|01-01-2026| 14560.7|14571.25| 14518.4|14560.25|3987031128|         47016.89|23.81| 3.7|     1.21|2026-01-01|
|           Nifty 500|01-01-2026|23914.65|23925.55|23835.55|23909.55|4549066214|   

In [135]:
double_cols = ["open","high","low","close","turnover_in_rs_cr","pe","pb","div_yield"]
df = df.withColumns({c: F.col(c).try_cast("double") for c in double_cols})
df.printSchema()
df.show()

root
 |-- index_name: string (nullable = true)
 |-- trade_date: string (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: long (nullable = true)
 |-- turnover_in_rs_cr: double (nullable = true)
 |-- pe: double (nullable = true)
 |-- pb: double (nullable = true)
 |-- div_yield: double (nullable = true)
 |-- index_date: date (nullable = true)

+--------------------+----------+--------+--------+--------+--------+----------+-----------------+-----+----+---------+----------+
|          index_name|trade_date|    open|    high|     low|   close|    volume|turnover_in_rs_cr|   pe|  pb|div_yield|index_date|
+--------------------+----------+--------+--------+--------+--------+----------+-----------------+-----+----+---------+----------+
|            Nifty 50|01-01-2026| 26173.3|26197.55| 26113.4|26146.55| 425631910|         23454.66|22.76|3.56|     1.28|2026-01-01|
|       

In [136]:
df.filter(F.col("volume").isNull()).show()

+--------------------+----------+--------+--------+--------+--------+------+-----------------+-----+----+---------+----------+
|          index_name|trade_date|    open|    high|     low|   close|volume|turnover_in_rs_cr|   pe|  pb|div_yield|index_date|
+--------------------+----------+--------+--------+--------+--------+------+-----------------+-----+----+---------+----------+
|Nifty50 Dividend ...|01-01-2026|    NULL|    NULL|    NULL|  249.14|  NULL|             NULL| NULL|NULL|     NULL|2026-01-01|
|           India VIX|01-01-2026|   9.475|   9.475|    8.72|    9.19|  NULL|             NULL| NULL|NULL|     NULL|2026-01-01|
|Nifty50 TR 1x Inv...|01-01-2026|   163.4|  163.75|  163.25|  163.55|  NULL|             NULL| NULL|NULL|     NULL|2026-01-01|
|Nifty50 TR 2x Lev...|01-01-2026|22067.35|22107.35|21967.95|22022.35|  NULL|             NULL| NULL|NULL|     NULL|2026-01-01|
|Nifty50 PR 1x Inv...|01-01-2026|   199.7|  200.15|  199.55|   199.9|  NULL|             NULL| NULL|NULL|     N

In [137]:
df.filter(F.col("volume").isNull()).count()

23

In [138]:
df.filter(F.col("close").isNull()).count()

0

In [139]:
df.filter(F.col("turnover_in_rs_cr").isNull()).count()

23

In [140]:
df.filter(F.col("pb").isNull()).count()

22

In [141]:
df.groupBy("index_name", "trade_date").count().filter(F.col("count") > 1).show()

+----------+----------+-----+
|index_name|trade_date|count|
+----------+----------+-----+
+----------+----------+-----+



In [143]:
df.filter(F.col("open").isNull()).count()

12

In [ ]:
df =df.na.drop(subset=["close"])